# We subsample 20 times, with replacement, and check if the fitting results are comparable to each other

In [ ]:
%run -i ~/project/preambles
%run -i ~/project/helper_functions
%run -i ~/project/fitting_functions

In [ ]:
# # Global parameters:
number_of_cycles = 500 # how many passes through the training data we go through
number_of_groups = 1 # divide the data set into smaller ones, to make fitting easier.
steps_per_batch = 5
dims = 2  # 2D spatial
head = 1000 # how many to down/subsample to.

In [ ]:
gene_list = [ "Inha", "Inhba", "Inhbb", "Fst", "Esr1", "Esr2", "Pgr", "Ar", "Cyp19a1", 
    "Cyp17a1", "Cyp11a1", "Lhcgr", "Parm1", "Akr1c18", "Fshr", "Star", "Ptgfr", 
    "Sfrp4", "Acvr1", "Acvr2a", "Acvr2b", "Ghr", "Lhb", "Cga"]
# gene_list = [ "Inha"]
# puck_list = 'all'
puck_list = ['Puck_230517_39'] # the largest puck

adata, X,Y,df,gene_list=load_data(gene_list=gene_list, head=head, puck_list = puck_list)
df

In [ ]:
head = 50 # how many locations to consider in the real data set.
estimated_params_df = pd.DataFrame()
for _ in range(25): # sub sampling with replacement the top head rows
    try:
        adata, X,Y,df=load_data(gene_list=gene_list, head=head, puck_list = puck_list)
        torch.autograd.set_detect_anomaly(True)
        optimized_marginal_params = optimize_marginal_parameters(X, Y, number_of_groups,  number_of_cycles, steps_per_batch)
        alpha_matrix, nu_matrix, sigma_matrix = optimize_cross_parameters(optimized_marginal_params,X,Y,number_of_groups,number_of_cycles,steps_per_batch)
        estimated_params_df = pd.concat([estimated_params_df, store_as_df(alpha_matrix, nu_matrix, sigma_matrix) ], ignore_index=True)
    except Exception as e:
        print(e)
        continue
estimated_params_df

In [ ]:
df_to_plot = estimated_params_df
histograms_are_plotted = False
%run -i epilogue